# 🦙 Colab Ollama Remote Inference Server

Run **Ollama** LLMs directly on Google Colab with GPU acceleration and expose an encrypted **Cloudflare Tunnel** for remote inference from your local Web UI, CLI, or AI agent.

> **Tip:** For maximum speed, ensure GPU acceleration is enabled via **Runtime > Change runtime type > T4 GPU** (or L4 / A100).

In [ ]:
# Step 1: Install Dependencies, Ollama & Cloudflared
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Ollama & Cloudflared installed successfully!")

In [ ]:
# Step 2: Configure & Start Ollama Background Service
import subprocess
import os
import time
import urllib.request

# Set environment variables for remote access
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OMP_NUM_THREADS"] = "8"

# Start daemon
subprocess.Popen(["ollama", "serve"])
time.sleep(3)

# Verify service
try:
    with urllib.request.urlopen("http://127.0.0.1:11434/") as response:
        print("✅ Status:", response.read().decode('utf-8'))
except Exception as e:
    print("❌ Failed to connect:", e)

In [ ]:
# Step 3: Pull Desired Models (e.g. qwen2.5-coder:1.5b, qwen2.5-coder:7b, llama3.2)
!ollama pull qwen2.5-coder:1.5b
!ollama list

In [ ]:
# Step 4: Start Cloudflare Tunnel for Remote Inference
import subprocess
import re
import time

subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--logfile", "/tmp/cf_tunnel.log"])
time.sleep(5)

tunnel_url = None
with open("/tmp/cf_tunnel.log", "r") as f:
    matches = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', f.read())
    if matches:
        tunnel_url = matches[0]

if tunnel_url:
    print("="*60)
    print(f"🚀 ACTIVE REMOTE INFERENCE URL: {tunnel_url}")
    print("="*60)
    print(f"\nAPI Endpoints:")
    print(f" • Base URL:      {tunnel_url}")
    print(f" • OpenAI API:    {tunnel_url}/v1")
    print(f" • Tags/Models:   {tunnel_url}/api/tags")
    print(f" • Generate:      {tunnel_url}/api/generate")
else:
    print("⚠️ Could not detect tunnel URL. Check /tmp/cf_tunnel.log.")